# GK2A 운량 나우캐스트 U-Net — v5 (+6시간 직접 예측)

**v4에서 바뀐 핵심**: `lc` 12 → **36**.

v4는 한 번에 최대 12스텝(=2시간)만 예측할 수 있어서, +6h를 만들려면 자기 예측을 다시
입력으로 넣는 **재귀**를 3번 돌렸습니다. 자기 예측은 실제 위성보다 흐릿해서 오차가 누적됐고,
검증에서 **정확히 2시간 지점부터 스킬이 꺾였습니다**(7월 +2h 18.4% → +3h 13.0%).
lc=36이면 **+6h까지 한 번에** 나오므로 재귀 자체가 사라집니다.

**그 밖의 강화**: 학습 데이터 대폭 확대(2025-01~2026-08 중 검증기간 제외 전부),
GPU에 맞춘 배치·혼합정밀, 짧은 주기 체크포인트, 예산 시간 상한.

**검증 격리(붙박이)**: 2025-10 / 2026-01 / 2026-04 / 2026-06-20~07-31 은 학습에서 자동 제외.

---

## 실행 방법 (Pay As You Go 100유닛 기준)

1. **런타임 → 런타임 유형 변경 → A100 GPU** 권장.
   유닛 소모는 A100 ~11.8/시간, L4 ~4.8/시간이라 **총 처리량은 비슷**하지만,
   PAYG에는 **백그라운드 실행이 없어** 브라우저를 닫으면 세션이 끊깁니다.
   같은 작업을 A100은 7시간, L4는 19시간에 하므로 **끊길 위험이 훨씬 낮은 A100이 유리**합니다.
2. **전체 실행** 후 **탭을 닫지 말고** 그대로 두세요(PC 절전도 꺼두면 좋습니다).
3. 학습은 `MAX_HOURS`(기본 7시간)에 도달하면 **스스로 멈추고 저장**합니다 — 유닛 초과 방지.
4. **중간에 끊겨도 안전합니다.** 800스텝마다 Drive에 체크포인트가 저장되고,
   다시 전체 실행하면 "체크포인트에서 재개"라고 뜨며 이어서 학습합니다.
   (더 학습할 필요가 없다고 판단되면, 재개 후 **학습 셀만 건너뛰고 저장 셀만** 실행해도 됩니다.)

**메모리**: 학습 프레임을 통째로 RAM에 올립니다(약 19GB). A100·L4 런타임은 여유가 있지만
T4 기본 런타임(12.7GB)에서는 부족합니다 — 아래 셀이 용량을 미리 알려줍니다.

In [ ]:
!pip install -q tf_keras
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/nwp_dl'
import os, glob, math, datetime as dt, shutil, time, json
import numpy as np
import tensorflow as tf
import tf_keras

gpus = tf.config.list_physical_devices('GPU')
name = tf.test.gpu_device_name()
det = tf.config.experimental.get_device_details(gpus[0]) if gpus else {}
cc = det.get('compute_capability', (0, 0))
gname = det.get('device_name', 'CPU')
# 배치: T4(16GB)=4, L4(24GB)=8, A100(40GB+)=16
BATCH = 16 if 'A100' in gname else (8 if ('L4' in gname or 'V100' in gname) else 4)
if cc[0] >= 7:
    tf_keras.mixed_precision.set_global_policy('mixed_float16')  # 속도 ~2배
    print('혼합정밀 사용')
print(f'GPU: {gname} | 배치 {BATCH} | TF {tf.__version__}')

In [ ]:
# 1) 모델 — from-scratch (사전학습 가중치 미사용). 입력 6채널은 lc와 무관하게 동일
#    ([hist4 | k/lc 스칼라평면 | 태양고도]) — 바뀌는 건 k의 범위뿐
N_LC = 36          # ← v4는 12. 36×10분 = 6시간 직접 예측
RESIZE_CONV = True # 체커보드 제거(아래 dec 참조). 구조가 바뀌므로
                   # v5 체크포인트와 호환되지 않는다 — 새로 학습해야 함
STEP, N_HIST = 10, 4
L = tf_keras.layers

def build_unet():
    inputs = L.Input((512, 512, 6))
    def cb(inp, n):
        x = L.Conv2D(n, 3, padding='same')(inp); x = L.BatchNormalization()(x); x = L.Activation('relu')(x)
        x = L.Conv2D(n, 3, padding='same')(x);   x = L.BatchNormalization()(x); return L.Activation('relu')(x)
    def enc(inp, n):
        x = cb(inp, n); return x, L.MaxPooling2D((2, 2))(x)
    def dec(inp, skip, n):
        if RESIZE_CONV:
            # 전치합성곱은 2px 체커보드를 남긴다(v5 실측: 경계비 1.09, 리드가 길수록 강해짐).
            # 표출 축소 과정에서 주기 14px 맥놀이로 커져 격자무늬로 보였다.
            # 업샘플+합성곱(resize-convolution)이 표준 해법 (Odena et al. 2016)
            x = L.UpSampling2D(2, interpolation='bilinear')(inp)
            x = L.Conv2D(n, 3, padding='same')(x)
        else:
            x = L.Conv2DTranspose(n, (2, 2), strides=2, padding='same')(inp)
        x = L.Concatenate()([x, skip]); return cb(x, n)
    s1, p1 = enc(inputs, 64); s2, p2 = enc(p1, 128); s3, p3 = enc(p2, 256); s4, p4 = enc(p3, 512)
    b1 = cb(p4, 1024)
    d1 = dec(b1, s4, 512); d2 = dec(d1, s3, 256); d3 = dec(d2, s2, 128); d4 = dec(d3, s1, 64)
    out = L.Conv2D(1, 1, padding='same', activation='sigmoid', dtype='float32')(d4)
    return tf_keras.Model(inputs, out)

CKPT = f"{BASE}/gk2a_{'v6' if RESIZE_CONV else 'v5'}_ckpt.h5"
model = build_unet()
if os.path.exists(CKPT):
    model.load_weights(CKPT); print('체크포인트에서 재개:', CKPT)
else:
    print('무작위 초기화(from scratch)')
print('파라미터', f'{model.count_params():,}')

In [ ]:
# 2) 데이터 — Drive→로컬 스테이징 후, 검증기간을 뺀 전부를 학습에 사용
import psutil
ram = psutil.virtual_memory().total / 2**30
print(f'런타임 RAM {ram:.0f}GB / 디스크 여유 {psutil.disk_usage("/content").free/2**30:.0f}GB')
if ram < 25:
    print('⚠ RAM이 부족할 수 있습니다(약 19GB 필요). A100 또는 L4 런타임을 쓰세요.')

LOCAL = '/content/dataset'
os.makedirs(LOCAL, exist_ok=True)
src = sorted(glob.glob(f'{BASE}/dataset/*.npz'))
print('Drive npz:', len(src))
for i, f in enumerate(src):
    dst = os.path.join(LOCAL, os.path.basename(f))
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(f):
        continue
    for a in range(5):
        try:
            shutil.copy2(f, dst); break
        except OSError as e:
            print('  재시도', os.path.basename(f), a + 1, e); time.sleep(3 * (a + 1))
    else:
        raise RuntimeError(f'복사 실패: {f}')
    if (i + 1) % 50 == 0:
        print(f'  스테이징 {i + 1}/{len(src)}')
print('스테이징 완료:', len(glob.glob(f'{LOCAL}/*.npz')))

# 검증(게이트) 기간 — 학습 절대 금지. 그 외 전부 학습.
GATE = [('20251001','20251031'), ('20260101','20260131'),
        ('20260401','20260430'), ('20260620','20260731')]
VAL  = [('20260819','20260821')]        # 학습 중 모니터링용(게이트 아님)

def in_ranges(day, ranges):
    return any(a <= day <= b for a, b in ranges)

def load_split(which):
    chunks, idx = [], {}
    for f in sorted(glob.glob(f'{LOCAL}/*.npz')):
        day = os.path.basename(f)[:8]
        if in_ranges(day, GATE):
            continue                      # 어느 split에도 넣지 않음
        is_val = in_ranges(day, VAL)
        if (which == 'val') != is_val:
            continue
        z = np.load(f)
        fi = len(chunks); chunks.append(z['frames'])
        for ri, s in enumerate(z['stamps']):
            idx[str(s)] = (fi, ri)
    n = sum(len(c) for c in chunks)
    print(f'  {which}: {len(chunks)}일 {n:,}프레임 ({n*512*512/2**30:.1f}GB)')
    return chunks, idx

fr_tr, IDX_TR = load_split('train')
fr_va, IDX_VA = load_split('val')

In [ ]:
# 3) 태양고도 채널 (고도각 도 단위 → 프레임별 min-max)
LATS = np.linspace(46.0, 29.7, 512)[:, None] * np.ones((1, 512))
LONS = np.ones((512, 1)) * np.linspace(113.0, 139.5, 512)[None, :]

def sun_channel(stamp):
    t = dt.datetime.strptime(stamp, '%Y%m%d%H%M')
    doy = t.timetuple().tm_yday
    decl = -23.44 * math.cos(math.radians(360/365*(doy+10)))
    ha = ((t.hour + t.minute/60)*15 - 180) + LONS
    sin_el = (np.sin(np.radians(LATS))*math.sin(math.radians(decl)) +
              np.cos(np.radians(LATS))*math.cos(math.radians(decl))*np.cos(np.radians(ha)))
    el = np.degrees(np.arcsin(np.clip(sin_el, -1, 1)))
    if el.max() > el.min():
        el = (el - el.min()) / np.ptp(el)
    return el.astype(np.float32)

In [ ]:
# 4) 샘플 생성기 — k는 0..N_LC-1 (즉 +10분 ~ +6시간)
def make_sample(frames, idx, s0, k):
    t0 = dt.datetime.strptime(s0, '%Y%m%d%H%M')
    need = [(t0 - dt.timedelta(minutes=STEP*i)).strftime('%Y%m%d%H%M') for i in range(N_HIST-1, -1, -1)]
    need.append((t0 + dt.timedelta(minutes=STEP*(k+1))).strftime('%Y%m%d%H%M'))
    if not all(n in idx for n in need):
        return None
    arrs = [frames[idx[n][0]][idx[n][1]] for n in need]
    if any((a == 255).mean() > 0.1 for a in arrs):
        return None
    hist = [np.where(a == 255, 50, a).astype(np.float32)/100.0 for a in arrs[:N_HIST]]
    y = np.where(arrs[-1] == 255, 50, arrs[-1]).astype(np.float32)/100.0
    lt = np.full((512, 512), k / N_LC, np.float32)
    return np.stack(hist + [lt, sun_channel(need[-1])], -1), y[..., None]

def gen(frames, idx, shuffle=True):
    """학습은 매번 무작위 k, 검증은 고정 k(n % N_LC).
    검증까지 k를 무작위로 뽑으면 매 에폭 다른 문제를 푸는 셈이라 val_mae가 노이즈가 되고,
    그 노이즈에 ReduceLROnPlateau가 반응해 학습률을 조기에 깎아버린다 (2026-08-26 실측:
    33에폭 만에 1e-4 → 6.25e-6). 검증은 반드시 결정적이어야 비교가 된다."""
    keys = list(idx.keys())
    while True:
        order = np.random.permutation(len(keys)) if shuffle else range(len(keys))
        for n, i in enumerate(order):
            k = int(np.random.randint(N_LC)) if shuffle else (n % N_LC)
            s = make_sample(frames, idx, keys[i], k)
            if s is not None:
                yield s

sig = (tf.TensorSpec((512, 512, 6), tf.float32), tf.TensorSpec((512, 512, 1), tf.float32))
ds_tr = (tf.data.Dataset.from_generator(lambda: gen(fr_tr, IDX_TR), output_signature=sig)
         .batch(BATCH).prefetch(tf.data.AUTOTUNE))
ds_va = (tf.data.Dataset.from_generator(lambda: gen(fr_va, IDX_VA, shuffle=False), output_signature=sig)
         .batch(BATCH).take(100))

In [ ]:
# 5) 학습 — 컴퓨트 유닛 예산에 맞춘 "시간 상한" 방식
#    Pay As You Go 100유닛 기준 소모율(대략): A100 ~11.8/h, L4 ~4.8/h, T4 ~1.8/h
#    → A100이면 7시간이 약 83유닛. 남는 유닛은 재개·검증용으로 남겨둔다.
MAX_HOURS = 7.0      # 이 시간에 도달하면 학습을 멈추고 저장 (유닛 보호)
STEPS     = 800      # 체크포인트 간격 — 짧게 잡아 끊겨도 손실 최소
EPOCHS    = 200      # 명목 상한. 실제로는 MAX_HOURS가 먼저 걸린다

def bcl1(y_true, y_pred):
    bc = tf_keras.losses.binary_crossentropy(y_true, y_pred)
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred), axis=-1)
    return bc + l1

class TimeStop(tf_keras.callbacks.Callback):
    """예산 시간 도달 시 안전 종료 + 경과/속도 보고."""
    def on_train_begin(self, logs=None):
        self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        el = (time.time() - self.t0) / 3600
        print(f'  [예산] 경과 {el:.2f}h / {MAX_HOURS}h '
              f'({(epoch + 1) * STEPS * BATCH:,}샘플 처리)')
        if el >= MAX_HOURS:
            print('  [예산] 시간 상한 도달 — 학습 종료')
            self.model.stop_training = True

model.compile(optimizer=tf_keras.optimizers.Adam(1e-4), loss=bcl1, metrics=['mae'])
cbs = [tf_keras.callbacks.ModelCheckpoint(CKPT, save_weights_only=True, save_freq='epoch'),
       # 보수적으로: 노이즈에 반응해 학습률이 조기에 무너지지 않도록 (min_lr도 1e-5로 올림)
       tf_keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=12,
                                            min_delta=0.002, min_lr=1e-5, verbose=1),
       TimeStop()]
hist = model.fit(ds_tr, steps_per_epoch=STEPS, epochs=EPOCHS,
                 validation_data=ds_va, callbacks=cbs)

In [ ]:
# 6) 저장 — 가중치 + 규격 사이드카(lc를 로컬 추론기가 읽음)
ver = 'v6' if RESIZE_CONV else 'v5'
out = f'{BASE}/gk2a_{ver}.h5'
model.save(out)
with open(f'{BASE}/gk2a_{ver}.json', 'w') as f:
    json.dump({'lc': N_LC, 'hist': N_HIST, 'step_min': STEP, 'size': 512}, f)
print('완료:', out, round(os.path.getsize(out)/2**20), 'MB')
print(f'→ gk2a_{ver}.h5 와 gk2a_{ver}.json 두 개를 PC의 kpx-model-charts/dl/ 에 내려받으세요')

In [ ]:
# 7) (선택) 눈검증 — 같은 입력에서 +1h / +3h / +6h 가 재귀 없이 한 번에
import matplotlib.pyplot as plt
keys = sorted(IDX_VA.keys()); s0 = keys[len(keys)//2]
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
base = make_sample(fr_va, IDX_VA, s0, 5)
ax[0].imshow(base[0][..., 3], cmap='gray', vmin=0, vmax=1); ax[0].set_title('input t'); ax[0].axis('off')
for a, k in zip(ax[1:], [5, 17, 35]):      # +60분, +180분, +360분
    s = make_sample(fr_va, IDX_VA, s0, k)
    if s is None: continue
    p = model.predict(s[0][None], verbose=0)[0, ..., 0]
    a.imshow(p, cmap='gray', vmin=0, vmax=1); a.set_title(f'pred +{(k+1)*10}min'); a.axis('off')
plt.show()